In [2]:
import pandas as pd
import numpy as np
import xgboost as xgb
import shap
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import roc_auc_score, classification_report
import joblib

# 1. ЗАГРУЗКА И ПОДГОТОВКА ДАННЫХ (OULAD)
print("Загрузка данных...")
info = pd.read_csv('studentInfo.csv')
assessments = pd.read_csv('studentAssessment.csv')
vle = pd.read_csv('studentVle.csv')

# Агрегируем оценки и активность
avg_scores = assessments.groupby('id_student')['score'].mean().reset_index()
avg_scores.rename(columns={'score': 'avg_test_score'}, inplace=True)

activity = vle.groupby('id_student')['sum_click'].sum().reset_index()
activity.rename(columns={'sum_click': 'total_clicks'}, inplace=True)

# Объединяем в один DataFrame
df = info[['id_student', 'final_result']].copy()
df = df.merge(avg_scores, on='id_student', how='left')
df = df.merge(activity, on='id_student', how='left')
df = df.fillna(0) # Заполняем пропуски нулями

# Создаем целевую переменную (1 - риск неуспеваемости, 0 - успешная сдача)
df['risk_of_failure'] = df['final_result'].apply(
    lambda x: 1 if x in ['Fail', 'Withdrawn'] else 0
)

# Удаляем текстовый статус, но пока ОСТАВЛЯЕМ id_student в df
df = df.drop(['final_result'], axis=1)

# 2. ПОДГОТОВКА ВЫБОРКИ ДЛЯ ОБУЧЕНИЯ
# Для обучения XGBoost выкидываем ID и целевую переменную
X = df.drop(['id_student', 'risk_of_failure'], axis=1)
y = df['risk_of_failure']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. ОБУЧЕНИЕ МОДЕЛИ ПРОГНОЗИРОВАНИЯ (XGBoost)
print("Обучение XGBoost...")
model = xgb.XGBClassifier(
    n_estimators=150, 
    max_depth=5, 
    learning_rate=0.05, 
    eval_metric='logloss',
    random_state=42
)
model.fit(X_train, y_train)

# Проверяем качество
y_prob = model.predict_proba(X_test)[:, 1]
print(f"Качество модели (ROC-AUC): {roc_auc_score(y_test, y_prob):.3f}")

# 4. ПОДГОТОВКА ОБЪЯСНЕНИЙ (SHAP)
print("Настройка SHAP...")
explainer = shap.TreeExplainer(model)

# 5. ОБУЧЕНИЕ МОДЕЛИ РЕКОМЕНДАЦИЙ (KNN)
print("Обучение системы рекомендаций (KNN)...")
# Берем только успешных студентов (risk_of_failure == 0)
# Вот здесь мы удаляем id_student и risk_of_failure, чтобы оставить только метрики
successful_students = df[df['risk_of_failure'] == 0].drop(['id_student', 'risk_of_failure'], axis=1)

knn = NearestNeighbors(n_neighbors=3, metric='cosine')
knn.fit(successful_students)

# 6. СОХРАНЕНИЕ ВСЕХ МОДЕЛЕЙ
print("Сохранение моделей в файлы...")
joblib.dump(model, 'xgboost_model.pkl')
joblib.dump(explainer, 'shap_explainer.pkl')
joblib.dump(knn, 'knn_model.pkl')

print("Готово! Модели успешно сохранены и готовы к использованию в FastAPI.")

Загрузка данных...
Обучение XGBoost...
Качество модели (ROC-AUC): 0.874
Настройка SHAP...
Обучение системы рекомендаций (KNN)...
Сохранение моделей в файлы...
Готово! Модели успешно сохранены и готовы к использованию в FastAPI.
